# 05 — DenseNet121-3D Single-Fold Sanity Check

**Purpose:** Quick single-fold sanity path to confirm the full pipeline works before launching the 10-fold CV (notebook 07).

- Uses `labels_v2.csv` (807 nodules, binary labels, min_rad≥3)
- Loads 40³ patches, random-crops to 32³ for train
- Modified DenseNet121 stem (3×3×3 s=1, no pool) to preserve 32³ spatial dims
- Patient-grouped 80/20 split (fold 0 of GroupKFold)
- CPU/MPS on Mac; targets CUDA on the lab machine

**After confirming this runs end-to-end without errors → run notebook 07 for full 10-fold CV.**

In [ ]:
import sys
import platform
import psutil
import torch

print(f"Python  : {sys.version}")
print(f"PyTorch : {torch.__version__}")
print(f"Platform: {platform.platform()}")


def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')


device = get_device()
print(f"Device  : {device}")

ram = psutil.virtual_memory()
print(f"RAM     : {ram.available / 1024**3:.1f} GB available / {ram.total / 1024**3:.1f} GB total")


In [ ]:
import sys
import subprocess
import importlib

required = {
    'monai': 'monai',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'yaml': 'pyyaml',
    'tqdm': 'tqdm',
}

for module, package in required.items():
    try:
        importlib.import_module(module)
        print(f"  OK : {package}")
    except ImportError:
        print(f"  Installing: {package}")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

try:
    import pytorch_grad_cam
    print('  OK : grad-cam')
except ImportError:
    print('  Installing: grad-cam')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'grad-cam', '-q'])

print("\nAll packages ready.")


In [ ]:
import sys
from pathlib import Path

# Resolve repo root without hardcoding any absolute path.
# find_repo_root() walks up from src/utils/paths.py until it finds a dir
# that contains both src/ and configs/.
_nb_dir = Path().resolve()
# Insert the repo root that contains src/ — works whether run from notebooks/
# or from the repo root directly.
_candidate = _nb_dir
while _candidate != _candidate.parent:
    if (_candidate / "src").is_dir() and (_candidate / "configs").is_dir():
        break
    _candidate = _candidate.parent
sys.path.insert(0, str(_candidate))

from src.utils.paths import find_repo_root, get_data_dir
REPO_ROOT = find_repo_root()
DATA_DIR  = get_data_dir(REPO_ROOT)

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"DATA_DIR  : {DATA_DIR}")
print(f"src exists: {(REPO_ROOT / 'src').is_dir()}")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

import yaml

from src.utils.device import get_device
from src.utils.dataloader import LIDCNoduleDataset, build_dataloaders, tta_predict
from src.models.densenet_3d import build_densenet, trace_feature_maps

print("All imports OK.")

In [ ]:
cfg_path = REPO_ROOT / "configs" / "densenet_config.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

device = get_device()

print(f"Config : {cfg_path}")
print(f"Device : {device}")
print(f"Batch  : {cfg['data']['batch_size']}")
print(f"Epochs : {cfg['training']['epochs']}")

In [ ]:
labels_path = DATA_DIR / cfg["data"]["labels_csv"]
df = pd.read_csv(labels_path)

print(f"Loaded {len(df)} nodules from {labels_path.name}")
print(f"  benign   (0): {(df['label']==0).sum()}")
print(f"  malignant(1): {(df['label']==1).sum()}")
print(f"  patients    : {df['patient_id'].nunique()}")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# Patient-grouped 80/20 split using first fold of GroupKFold(10).
# Same splitter as the full CV (notebook 07) — fold 0 val set is a real held-out set.

gkf = GroupKFold(n_splits=10)
groups = df["patient_id"].values
splits = list(gkf.split(df, df["label"], groups))
train_idx, val_idx = splits[0]

train_df = df.iloc[train_idx].copy()
val_df   = df.iloc[val_idx].copy()

# Zero-leakage assertion
train_pts = set(train_df["patient_id"])
val_pts   = set(val_df["patient_id"])
overlap = train_pts & val_pts
assert len(overlap) == 0, f"Patient leakage: {overlap}"

print(f"Train: {len(train_df)} nodules / {len(train_pts)} patients")
print(f"Val  : {len(val_df)} nodules  / {len(val_pts)} patients")
print("Zero-leakage check: PASSED")

In [ ]:
train_loader, val_loader = build_dataloaders(cfg, train_df, val_df)
print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

In [ ]:
model = build_densenet(cfg).to(device)

# Verify stem fix: run shape trace to confirm 32→32 through stem (not 32→8)
# Comment out on lab machine after first verification to save time.
# trace_feature_maps(model, device)

In [ ]:
train_cfg = cfg["training"]
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=train_cfg["lr"],
    weight_decay=train_cfg.get("weight_decay", 1e-4),
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=train_cfg.get("lr_step_size", 30),
    gamma=train_cfg.get("lr_gamma", 0.5),
)
criterion = nn.CrossEntropyLoss()

EPOCHS   = train_cfg["epochs"]
use_amp  = train_cfg.get("amp", False) and device.type == "cuda"
amp_dtype = torch.bfloat16 if train_cfg.get("amp_dtype", "bfloat16") == "bfloat16" else torch.float16

print(f"Optimizer : Adam lr={train_cfg['lr']} wd={train_cfg.get('weight_decay',1e-4)}")
print(f"Scheduler : StepLR step={train_cfg.get('lr_step_size',30)} gamma={train_cfg.get('lr_gamma',0.5)}")
print(f"Epochs    : {EPOCHS}  |  AMP: {use_amp} ({amp_dtype})")

In [ ]:
print("Sanity check — one train batch + one val batch ...")
model.eval()

x, y = next(iter(train_loader))
print(f"  Train batch: x={x.shape} y={y.shape} dtype={x.dtype}")
assert x.shape == (cfg["data"]["batch_size"], 1, 32, 32, 32) or x.shape[1:] == (1, 32, 32, 32), \
    f"Unexpected shape: {x.shape}"

with torch.no_grad():
    logits = model(x.to(device))
print(f"  Logits: {logits.shape}  loss={criterion(logits, y.to(device)).item():.4f}")

x_v, y_v = next(iter(val_loader))
probs = tta_predict(model, x_v, cfg.get("tta_transforms", ["identity"]), device)
print(f"  Val TTA probs: {probs.shape}")

model.train()
print("Sanity check PASSED.")

In [ ]:
save_dir = REPO_ROOT / cfg["training"]["save_dir"]
save_dir.mkdir(parents=True, exist_ok=True)

patience         = cfg["training"].get("early_stopping_patience", 40)
best_val_auc     = -1.0
epochs_no_improve = 0
history          = []

tta_transforms = cfg.get("tta_transforms", ["identity"]) if cfg.get("tta", False) else ["identity"]

for epoch in range(EPOCHS):
    t0 = time.time()
    # ── Train ──
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
            logits = model(x)
            loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    # ── Val ──
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            probs = tta_predict(model, x, tta_transforms, device)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(y.numpy())

    import numpy as np
    y_prob = np.concatenate(all_probs)[:, 1]
    y_true = np.concatenate(all_labels)
    val_auc = float(roc_auc_score(y_true, y_prob))
    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1:03d}/{EPOCHS}  loss={avg_loss:.4f}  val_auc={val_auc:.4f}  ({time.time()-t0:.0f}s)")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        epochs_no_improve = 0
        torch.save({"epoch": epoch+1, "model_state_dict": model.state_dict(), "val_auc": val_auc},
                   save_dir / "sanity_best.pth")
    else:
        epochs_no_improve += 1

    history.append({"epoch": epoch+1, "train_loss": avg_loss, "val_auc": val_auc})

    if epochs_no_improve >= patience:
        print(f"Early stop at epoch {epoch+1}. Best AUC={best_val_auc:.4f}")
        break

print(f"\nBest AUC: {best_val_auc:.4f}  checkpoint: {save_dir/'sanity_best.pth'}")

In [ ]:
import pandas as pd
history_df = pd.DataFrame(history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history_df["epoch"], history_df["train_loss"], "b-o", ms=3)
ax1.set(xlabel="Epoch", ylabel="CE Loss", title="Train Loss")
ax1.grid(alpha=0.3)

ax2.plot(history_df["epoch"], history_df["val_auc"], "r-o", ms=3, label="Val AUC")
ax2.axhline(best_val_auc, ls="--", color="gray", alpha=0.5, label=f"Best {best_val_auc:.4f}")
ax2.set(xlabel="Epoch", ylabel="AUC", title="Val AUC", ylim=(0, 1.05))
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("DenseNet121-3D Sanity Fold", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)

ckpt = torch.load(save_dir / "sanity_best.pth", map_location=device, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for x, y in val_loader:
        probs = tta_predict(model, x, tta_transforms, device)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.numpy())

import numpy as np
y_prob = np.concatenate(all_probs)[:, 1]
y_true = np.concatenate(all_labels)
y_pred = (y_prob >= 0.5).astype(int)

tn = int(((y_pred==0)&(y_true==0)).sum())
fp = int(((y_pred==1)&(y_true==0)).sum())
spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print("=== Sanity Fold — Final Metrics (best checkpoint, TTA) ===")
print(f"  AUC          : {roc_auc_score(y_true, y_prob):.4f}")
print(f"  Accuracy     : {accuracy_score(y_true, y_pred):.4f}")
print(f"  Sensitivity  : {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  Specificity  : {spec:.4f}")
print(f"  Precision    : {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  F1           : {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"\n  Best epoch   : {ckpt['epoch']}")
print("=== End. If metrics look reasonable → run notebook 07 for full 10-fold CV ===")